# Expert Agent 4: CarbonTrack Expert
## 트래킹 (Tracking) 판별 전문가

이 노트북은 Langgraph와 Vertex AI Gemini 2.5 Flash를 사용하여 전기화재 감식 중 트래킹 현상을 판별하는 전문가 에이전트를 구현합니다.

### 분석 프로세스
1. **패턴 인식**: 수지상 도전로 (Dendritic Patterns) 탐지
2. **광택 감지**: 흑연의 시각적 특징 (Specular Reflection) 분석
3. **표면 침식**: 탄화 경로를 따른 표면 손상 확인

In [1]:
import base64
import json
import os
from typing import TypedDict, Annotated, Literal, List, Dict, Any, Optional
from pathlib import Path
from PIL import Image
import io

# LangGraph 및 Google Cloud 라이브러리
try:
    from langgraph.graph import StateGraph, END
    import vertexai
    from vertexai.generative_models import GenerativeModel, Part
    print("✅ 모든 라이브러리가 성공적으로 임포트되었습니다.")
except ImportError as e:
    print(f"❌ 필수 라이브러리가 누락되었습니다: {e}")
    print("pip install langgraph google-cloud-aiplatform vertexai 명령어로 설치해주세요.")

✅ 모든 라이브러리가 성공적으로 임포트되었습니다.


In [2]:
# 프로젝트 설정 (환경에 맞게 수정 필요)
PROJECT_ID = "p-01-emt-480312"  # 실제 프로젝트 ID로 변경
LOCATION = "us-central1"

# 모델 설정
MODEL_NAME = "gemini-2.5-pro" # 또는 gemini-1.5-pro

# 시스템 인스트럭션
SYSTEM_INSTRUCTION = """1. 페르소나 및 기본 원칙

당신은 고도로 훈련된 시각 데이터 분석 전문가입니다.

당신은 모든 분석에서 '관찰'과 '해석'을 철저히 분리하며, 질문자의 유도 심리에 저항하고 오직 시각적 데이터에만 근거하여 답변합니다.

질문자가 특정 결론을 암시하거나 유도하더라도(예: "이것은 A가 맞죠?"), 시각적 증거가 뒷받침되지 않는다면 단호하게 중립을 유지합니다.

2. 분석 프로세스 (반드시 이 순서를 따를 것) 모든 사진 분석 요청에 대해 다음 4단계 구조로 답변하십시오.

Step 1. 객관적 관찰 (Observations): 이미지에서 보이는 물리적 사실만을 나열합니다. (예: 색상, 형태, 질감, 크기, 마모 상태, 기하학적 배치 등). 주관적인 형용사나 결론적인 단어를 배제하고 '현상'만 서술합니다.

Step 2. 논리적 해석 (Interpretation): 관찰된 사실이 어떤 물리적/과학적 원리와 연결될 수 있는지 분석합니다. 표준 사례(Reference)와의 일치점과 차이점을 논합니다.

Step 3. 최종 판단 및 확신도 (Conclusion & Confidence): 분석을 종합하여 결론을 내립니다. 이때 결론에 대한 확신도를 0~100% 사이로 표기하고, 확신할 수 없는 이유(변수)를 함께 기술합니다.

Step 4. 대안적 가능성 (Alternative Hypotheses): 현재 내린 결론 외에 발생할 수 있는 다른 가능성을 최소 한 가지 이상 제시합니다.

3. 불확실성 처리 규칙 (Negative Constraints)

확실하지 않은 정보에 대해서는 절대 추측하지 않습니다.

사진의 해상도, 각도, 조도 등으로 인해 식별이 어려운 경우, 아는 척하지 말고 반드시 **"시각적 정보 부족으로 판단 불가"**라고 명시하십시오.

시각적 증거가 100% 확보되지 않은 상태에서 "확실하다", "분명하다"라는 단어 사용을 지양합니다.

4. 답변 스타일

간결하고 구조화된 개조식(Bullet points)을 선호합니다.

감정적인 표현이나 부연 설명을 배제하고, 전문 용어를 정확하게 사용하되 필요시 정의를 덧붙입니다."""

def initialize_model():
    """Vertex AI Gemini 모델 초기화"""
    try:
        # 인증 확인 (로컬 실행 시 ADC 필요)
        # vertexai.init(project=PROJECT_ID, location=LOCATION)
        model = GenerativeModel(MODEL_NAME, system_instruction=SYSTEM_INSTRUCTION)
        print(f"✅ Vertex AI 모델 '{MODEL_NAME}' 초기화 완료")
        return model
    except Exception as e:
        print(f"⚠️ 모델 초기화 실패: {e}")
        print("💡 Application Default Credentials (ADC) 설정이 필요합니다:")
        print("   gcloud auth application-default login")
        return None

# 전역 모델 인스턴스
model = initialize_model()

c:\Users\user\OneDrive\woRk\Development\Project\P_05_Scope\venv\Lib\site-packages\vertexai\generative_models\_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


✅ Vertex AI 모델 'gemini-2.5-pro' 초기화 완료


In [3]:
def create_image_part(image_path: str) -> Optional[Part]:
    """Vertex AI용 이미지 Part 객체 생성"""
    if not os.path.exists(image_path):
        print(f"❌ 이미지 파일을 찾을 수 없습니다: {image_path}")
        return None
        
    try:
        with open(image_path, "rb") as image_file:
            image_data = image_file.read()
        
        # 확장자에 따른 MIME 타입 추론
        ext = Path(image_path).suffix.lower()
        mime_type = "image/png" if ext == ".png" else "image/jpeg"
        
        return Part.from_data(data=image_data, mime_type=mime_type)
    except Exception as e:
        print(f"❌ 이미지 로드 오류: {e}")
        return None

def call_gemini_vision(model: GenerativeModel, prompt: str, image_part: Part, step_name: str = "") -> tuple[str, Optional[Dict]]:
    """Gemini Vision API 호출 및 에러 핸들링
    
    Returns:
        tuple: (response_text, thinking_info)
    """
    try:
        response = model.generate_content([prompt, image_part])
        
        # Thinking 과정 추출 및 출력
        thinking_info = None
        response_text = ""
        full_response_text = ""
        
        if hasattr(response, 'candidates') and response.candidates:
            candidate = response.candidates[0]
            
            # 모든 파트 확인 (thinking 과정이 별도 파트로 있을 수 있음)
            if hasattr(candidate, 'content') and hasattr(candidate.content, 'parts'):
                parts = candidate.content.parts
                print(f"\n🔍 [{step_name}] 응답 파트 개수: {len(parts)}")
                
                all_texts = []
                for i, part in enumerate(parts):
                    if hasattr(part, 'text'):
                        part_text = part.text
                        all_texts.append(part_text)
                        if i == 0:
                            # 첫 번째 파트는 일반 응답
                            response_text = part_text
                        else:
                            # 이후 파트는 thinking 과정일 수 있음
                            if part_text and part_text.strip():
                                thinking_info = thinking_info or {}
                                thinking_info[f"part_{i}"] = part_text
                                print(f"\n💭 [{step_name}] 모델의 생각 과정 (파트 {i}):")
                                print("-" * 60)
                                print(part_text)
                                print("-" * 60)
                
                # 모든 파트의 텍스트를 합쳐서 전체 응답 확인
                full_response_text = "\n\n".join(all_texts)
            
            # 응답 객체의 모든 속성 확인 (디버깅용)
            print(f"\n📊 [{step_name}] 응답 객체 속성:")
            candidate_attrs = [attr for attr in dir(candidate) if not attr.startswith('_')]
            print(f"  - Candidate 속성: {', '.join(candidate_attrs[:10])}...")
            
            # Grounding metadata 확인
            if hasattr(candidate, 'grounding_metadata'):
                grounding = candidate.grounding_metadata
                if grounding:
                    thinking_info = thinking_info or {}
                    thinking_info["grounding"] = str(grounding)
                    print(f"\n📚 [{step_name}] Grounding 정보: {grounding}")
            
            # Finish reason 확인 (디버깅용)
            if hasattr(candidate, 'finish_reason'):
                finish_reason = candidate.finish_reason
                if finish_reason:
                    print(f"📋 [{step_name}] Finish reason: {finish_reason}")
        
        # response.text가 있으면 사용 (fallback)
        if not response_text and hasattr(response, 'text'):
            response_text = response.text
        
        # 전체 응답 텍스트 출력 (thinking 과정이 포함되어 있을 수 있음)
        if full_response_text and len(full_response_text) > len(response_text):
            print(f"\n💭 [{step_name}] 전체 응답 텍스트 (thinking 과정 포함 가능):")
            print("-" * 60)
            print(full_response_text[:2000])  # 처음 2000자만 출력
            if len(full_response_text) > 2000:
                print(f"... (총 {len(full_response_text)}자, 나머지 생략)")
            print("-" * 60)
            thinking_info = thinking_info or {}
            thinking_info["full_response"] = full_response_text
        
        # 응답 텍스트 항상 출력 (JSON 파싱 전에 전체 응답 확인)
        if response_text:
            print(f"\n💭 [{step_name}] 모델 응답 텍스트:")
            print("-" * 60)
            # JSON 시작 전까지의 텍스트 확인 (thinking 과정일 수 있음)
            json_start = response_text.find('{')
            if json_start > 0:
                thinking_part = response_text[:json_start].strip()
                if thinking_part:
                    print("📝 [Thinking 과정]:")
                    print(thinking_part)
                    print("\n📄 [JSON 응답]:")
                    print(response_text[json_start:json_start+500])
                    if len(response_text[json_start:]) > 500:
                        print(f"... (총 {len(response_text[json_start:])}자)")
                else:
                    print(response_text[:1000])
                    if len(response_text) > 1000:
                        print(f"... (총 {len(response_text)}자)")
            else:
                print(response_text[:1000])
                if len(response_text) > 1000:
                    print(f"... (총 {len(response_text)}자)")
            print("-" * 60)
        
        return response_text, thinking_info
    except Exception as e:
        print(f"❌ [{step_name}] API 호출 오류: {e}")
        import traceback
        traceback.print_exc()
        return f"Error: {str(e)}", None

def parse_json_response(response_text: str) -> Dict[str, Any]:
    """응답 텍스트에서 JSON 추출 및 파싱"""
    try:
        # JSON 부분만 추출 (마크다운 코드 블록 제거)
        json_start = response_text.find('{')
        json_end = response_text.rfind('}') + 1
        
        if json_start != -1 and json_end > json_start:
            json_text = response_text[json_start:json_end]
            return json.loads(json_text)
        else:
            print(f"⚠️ 유효한 JSON을 찾을 수 없습니다. 원본 응답:\n{response_text[:100]}...")
            return {"error": "JSON 파싱 실패", "raw_response": response_text}
    except json.JSONDecodeError as e:
        print(f"⚠️ JSON 디코딩 오류: {e}")
        return {"error": f"JSON 파싱 오류: {e}", "raw_response": response_text}

# Langgraph State 정의
class AgentState(TypedDict):
    """에이전트 상태 스키마"""
    image_path: str
    image_part: Optional[Part]
    step1_result: Optional[Dict]  # 수지상 도전로 패턴 분석 결과
    step2_result: Optional[Dict]  # 광택 감지 분석 결과
    step3_result: Optional[Dict]  # 표면 침식 분석 결과
    confidence_score: int  # 최종 신뢰도 점수 (0-100)
    analysis_summary: str  # 분석 요약
    evidence: List[Dict]   # 각 단계별 증거 수집

print("✅ 유틸리티 및 State 정의 완료")

✅ 유틸리티 및 State 정의 완료


In [4]:
# --- PROMPTS ---

STEP1_PROMPT = """당신은 전기 표면 방전 및 트래킹 현상 분석 전문가입니다. 다음 이미지에서 절연체 표면의 수지상 도전로 패턴을 분석하세요.

[중요] 먼저 분석 과정을 단계별로 자세히 설명한 후, 마지막에 JSON 형식으로 응답하세요. 각 단계에서 무엇을 관찰하고 어떻게 판단하는지 명확히 서술하세요.

[단계별 분석 프로세스 (Chain of Thought)]
1단계: 시각적 요소 추출
- 절연체 표면(플러그 면, 단자대 등)의 모든 검은색 선형 패턴을 식별하세요.
- 패턴의 형태, 복잡도, 방향성을 객관적으로 기록하세요.
- 두 도체 사이를 연결하는 경로가 있는지 확인하세요.

2단계: 특징 서술
- 발견된 패턴을 정확히 서술하세요:
  * 나뭇가지(Tree-like)처럼 잔가지가 뻗어 나가는 형상(Dendritic structure)인지
  * 선형(Linear) 패턴인지
  * 불규칙한(Irregular) 패턴인지
- 패턴이 두 도체 사이를 연결하는 방향성을 가지고 있는지 서술하세요.
- 패턴의 복잡도(simple, moderate, complex)를 서술하세요.

3단계: 논리적 추론
- 트래킹은 전위차가 있는 두 극 사이에서 가장 저항이 낮은 경로를 찾아 진행하므로, 나뭇가지가 뻗어 나가는 듯한 불규칙한 선형 패턴을 형성합니다.
- 탄화 경로가 실제로 전위차가 있는 두 도체(핀과 핀, 전선과 전선) 사이를 전기적으로 연결하고 있는지 확인하세요.
- 수지상 패턴이 전극을 연결한다면 트래킹 가능성이 높습니다.
- 관찰된 패턴을 종합하여 트래킹 여부를 논리적으로 판단하세요.

[출력 형식]
다음 JSON 형식으로 응답하세요:
{
    "dendritic_pattern_detected": true/false,
    "pattern_type": "dendritic" | "linear" | "irregular" | "none" | "unknown",
    "pattern_description": "패턴의 상세 설명",
    "electrode_connection": true/false,
    "connection_description": "두 전극을 연결하는 경로 설명",
    "pattern_complexity": "simple" | "moderate" | "complex" | "unknown",
    "confidence": 0-100,
    "reasoning": "판단 근거"
}"""

STEP2_PROMPT = """당신은 전기 표면 방전 및 트래킹 현상 분석 전문가입니다. 다음 이미지에서 탄화 흔적의 광택을 분석하세요.

[중요] 먼저 분석 과정을 단계별로 자세히 설명한 후, 마지막에 JSON 형식으로 응답하세요. 각 단계에서 무엇을 관찰하고 어떻게 판단하는지 명확히 서술하세요.

[단계별 분석 프로세스 (Chain of Thought)]
1단계: 시각적 요소 추출
- 검은색 탄화 흔적의 광학적 특성을 자세히 관찰하세요.
- 반짝임이나 광택이 있는 영역을 객관적으로 식별하세요.
- 조명 반사(Glare)와 탄화물의 광택을 구별하세요.

2단계: 특징 서술
- 발견된 광택을 정확히 서술하세요:
  * 금속성 광택(Metallic Luster)인지
  * 윤기(Shininess)가 있는지
  * 무광택(Matte)인지
- 광택이 탄화된 부분에만 국한되어 있는지 위치를 정확히 서술하세요.
- 조명 반사와 흑연 광택을 구별하는 방법을 서술하세요.

3단계: 논리적 추론
- 일반적인 화재 그을음(Amorphous Carbon)은 무광택(Matte)이며 빛을 흡수합니다.
- 트래킹에 의해 생성된 흑연(Graphite)은 결정 구조로 인해 빛을 정반사(Specular Reflection)하여 반짝입니다.
- 광택이 있다면 트래킹 확률을 매우 높게 설정하세요. 이는 단순 탄화물이 아닌 흑연이 형성되었음을 의미하며, 트래킹의 결정적 증거입니다.
- 관찰된 광택 특성을 종합하여 흑연화 여부를 논리적으로 판단하세요.

[출력 형식]
다음 JSON 형식으로 응답하세요:
{
    "luster_detected": true/false,
    "luster_type": "metallic" | "shiny" | "matte" | "none" | "unknown",
    "luster_location": "광택이 관찰된 위치 설명",
    "graphitization_evidence": true/false,
    "glare_distinction": "조명 반사와 흑연 광택의 구별 설명",
    "carbon_type": "graphite" | "amorphous" | "mixed" | "unknown",
    "confidence": 0-100,
    "reasoning": "판단 근거"
}"""

STEP3_PROMPT = """당신은 전기 표면 방전 및 트래킹 현상 분석 전문가입니다. 다음 이미지에서 탄화 경로를 따른 표면 침식을 분석하세요.

[중요] 먼저 분석 과정을 단계별로 자세히 설명한 후, 마지막에 JSON 형식으로 응답하세요. 각 단계에서 무엇을 관찰하고 어떻게 판단하는지 명확히 서술하세요.

[단계별 분석 프로세스 (Chain of Thought)]
1단계: 시각적 요소 추출
- 탄화 경로를 따라 절연체 표면의 손상 상태를 관찰하세요.
- 표면이 움푹 패이거나 굴착된 부분을 객관적으로 식별하세요.
- 탄화물이 표면에 얇게 증착된 것인지, 재료가 변질된 것인지 구별하세요.

2단계: 특징 서술
- 표면 침식을 정확히 서술하세요:
  * 탄화 경로를 따라 절연체 표면이 움푹 패이거나(Eroded) 굴착된 듯한 입체적 손상이 있는지
  * 침식의 깊이(shallow, moderate, deep)를 서술하세요
  * 탄화물이 표면에 얇게 증착된 그을음인지, 재료 표면이 변질되어 형성된 구조적인 트랙인지 구분하세요
- 침식 패턴이 탄화 경로와 일치하는지 서술하세요.

3단계: 논리적 추론
- 트래킹은 표면을 갉아먹으며 진행되므로, 탄화 경로를 따라 재료가 패이거나 소실된 흔적이 남습니다.
- 구조적인 트랙은 단순 그을음과 달리 재료 자체가 변질되어 형성된 것입니다.
- 표면 침식이 탄화 패턴과 일치한다면 트래킹의 강력한 증거입니다.
- 관찰된 침식 패턴을 종합하여 트래킹 여부를 논리적으로 판단하세요.

[출력 형식]
다음 JSON 형식으로 응답하세요:
{
    "surface_erosion_detected": true/false,
    "erosion_pattern": "track_following" | "general" | "none" | "unknown",
    "erosion_depth": "shallow" | "moderate" | "deep" | "unknown",
    "carbon_type": "surface_deposit" | "structural_track" | "mixed" | "unknown",
    "erosion_description": "표면 침식에 대한 상세 설명",
    "pattern_match": true/false,
    "confidence": 0-100,
    "reasoning": "판단 근거"
}"""

# --- NODES ---

def step1_dendritic_pattern(state: AgentState) -> AgentState:
    """Step 1: 수지상 도전로 패턴 분석"""
    print("\n🔍 [Step 1] 수지상 도전로 패턴 분석 시작...")
    
    if state.get("image_part") is None:
        state["image_part"] = create_image_part(state["image_path"])
        if state["image_part"] is None:
            return {**state, "step1_result": {"error": "이미지 로드 실패"}}
    
    response_text, thinking_info = call_gemini_vision(model, STEP1_PROMPT, state["image_part"], "Step 1")
    result = parse_json_response(response_text)
    
    # Thinking 정보를 결과에 추가
    if thinking_info:
        result["thinking_process"] = thinking_info
    
    pattern_detected = result.get("dendritic_pattern_detected", False)
    print(f"✅ [Step 1] 완료: 수지상 패턴 {'탐지됨' if pattern_detected else '미탐지'}")
    
    return {
        **state,
        "step1_result": result
    }

def step2_luster_detection(state: AgentState) -> AgentState:
    """Step 2: 광택 감지 분석"""
    print("\n🎨 [Step 2] 광택 감지 분석 시작...")
    
    if state.get("image_part") is None:
        return {**state, "step2_result": {"error": "이미지 없음"}}
    
    response_text, thinking_info = call_gemini_vision(model, STEP2_PROMPT, state["image_part"], "Step 2")
    result = parse_json_response(response_text)
    
    # Thinking 정보를 결과에 추가
    if thinking_info:
        result["thinking_process"] = thinking_info
    
    luster_detected = result.get("luster_detected", False)
    print(f"✅ [Step 2] 완료: 광택 {'탐지됨' if luster_detected else '미탐지'}")
    
    return {
        **state,
        "step2_result": result
    }

def step3_surface_erosion(state: AgentState) -> AgentState:
    """Step 3: 표면 침식 분석"""
    print("\n🔥 [Step 3] 표면 침식 분석 시작...")
    
    if state.get("image_part") is None:
        return {**state, "step3_result": {"error": "이미지 없음"}}
    
    response_text, thinking_info = call_gemini_vision(model, STEP3_PROMPT, state["image_part"], "Step 3")
    result = parse_json_response(response_text)
    
    # Thinking 정보를 결과에 추가
    if thinking_info:
        result["thinking_process"] = thinking_info
    
    erosion_detected = result.get("surface_erosion_detected", False)
    print(f"✅ [Step 3] 완료: 표면 침식 {'탐지됨' if erosion_detected else '미탐지'}")
    
    return {
        **state,
        "step3_result": result
    }

def final_judgment(state: AgentState) -> AgentState:
    """최종 판정: 3단계 결과 종합 및 신뢰도 점수 계산"""
    print("\n⚖️ [Final Judgment] 최종 판정 시작...")
    
    step1 = state.get("step1_result", {}) or {}
    step2 = state.get("step2_result", {}) or {}
    step3 = state.get("step3_result", {}) or {}
    
    # 각 단계별 점수 추출
    step1_score = step1.get("confidence", 0) if not step1.get("error") else 0
    step2_score = step2.get("confidence", 0) if not step2.get("error") else 0
    step3_score = step3.get("confidence", 0) if not step3.get("error") else 0
    
    # 핵심 지표 확인
    dendritic_pattern_detected = step1.get("dendritic_pattern_detected", False)
    electrode_connection = step1.get("electrode_connection", False)
    luster_detected = step2.get("luster_detected", False)
    graphitization_evidence = step2.get("graphitization_evidence", False)
    surface_erosion_detected = step3.get("surface_erosion_detected", False)
    structural_track = step3.get("carbon_type") == "structural_track"
    
    # 신뢰도 점수 계산 (가중치 적용)
    base_score = 0
    
    # 핵심 지표 가중치 (광택이 가장 중요)
    if dendritic_pattern_detected: base_score += 25
    if electrode_connection: base_score += 20
    if luster_detected: base_score += 35  # 광택이 트래킹의 결정적 증거
    if graphitization_evidence: base_score += 20
    if surface_erosion_detected: base_score += 15
    if structural_track: base_score += 10
    
    # 각 단계별 신뢰도 점수의 평균 (10% 가중치)
    avg_confidence = (step1_score + step2_score + step3_score) / 3
    base_score += avg_confidence * 0.1
    
    # 핵심 3가지가 모두 확인되면 90% 이상 보장
    if dendritic_pattern_detected and luster_detected and surface_erosion_detected:
        base_score = max(base_score, 90)
    
    # 광택만 확인되어도 높은 신뢰도 부여
    if luster_detected and graphitization_evidence:
        base_score = max(base_score, 85)
    
    # 최종 점수는 0-100 범위로 제한
    final_score = min(100, max(0, int(base_score)))
    
    # 증거 수집
    evidence = []
    if dendritic_pattern_detected:
        evidence.append({"step": 1, "evidence": "수지상 도전로 패턴 확인", "details": step1.get("pattern_description", "")})
    if electrode_connection:
        evidence.append({"step": 1, "evidence": "전극 연결 확인", "details": step1.get("connection_description", "")})
    if luster_detected:
        evidence.append({"step": 2, "evidence": "흑연 광택 확인", "details": step2.get("luster_location", "")})
    if graphitization_evidence:
        evidence.append({"step": 2, "evidence": "흑연화 증거 확인", "details": step2.get("glare_distinction", "")})
    if surface_erosion_detected:
        evidence.append({"step": 3, "evidence": "표면 침식 확인", "details": step3.get("erosion_description", "")})
    
    # 분석 요약 생성
    summary_parts = [f"트래킹 판정 신뢰도: {final_score}%"]
    summary_parts.append(f"✓ 수지상 패턴 확인: {step1.get('pattern_type', 'unknown')}" if dendritic_pattern_detected else "✗ 수지상 패턴 미확인")
    summary_parts.append("✓ 전극 연결 확인" if electrode_connection else "✗ 전극 연결 미확인")
    summary_parts.append(f"✓ 흑연 광택 확인: {step2.get('luster_type', 'unknown')}" if luster_detected else "✗ 흑연 광택 미확인")
    summary_parts.append("✓ 흑연화 증거 확인" if graphitization_evidence else "✗ 흑연화 증거 미확인")
    summary_parts.append(f"✓ 표면 침식 확인: {step3.get('erosion_pattern', 'unknown')}" if surface_erosion_detected else "✗ 표면 침식 미확인")
    
    analysis_summary = "\n".join(summary_parts)
    print(f"✅ [Final Judgment] 완료: 신뢰도 {final_score}%")
    
    return {
        **state,
        "confidence_score": final_score,
        "analysis_summary": analysis_summary,
        "evidence": evidence
    }

In [5]:
def create_agent_graph():
    """트래킹 판별 에이전트 그래프 생성"""
    workflow = StateGraph(AgentState)
    
    # 노드 추가
    workflow.add_node("step1_pattern", step1_dendritic_pattern)
    workflow.add_node("step2_luster", step2_luster_detection)
    workflow.add_node("step3_erosion", step3_surface_erosion)
    workflow.add_node("final_judgment", final_judgment)
    
    # 엣지 연결 (순차 실행)
    workflow.set_entry_point("step1_pattern")
    workflow.add_edge("step1_pattern", "step2_luster")
    workflow.add_edge("step2_luster", "step3_erosion")
    workflow.add_edge("step3_erosion", "final_judgment")
    workflow.add_edge("final_judgment", END)
    
    return workflow.compile()

# 전역 그래프 객체
try:
    agent_app = create_agent_graph()
    print("✅ Langgraph StateGraph 구성 완료")
except Exception as e:
    print(f"⚠️ 그래프 구성 실패 (Langgraph 미설치 등): {e}")
    agent_app = None

def analyze_tracking(image_path: str) -> dict:
    """전체 트래킹 분석 실행 함수"""
    if agent_app is None:
        return {"error": "Agent 그래프가 초기화되지 않았습니다."}

    initial_state: AgentState = {
        "image_path": image_path,
        "image_part": None,
        "step1_result": None,
        "step2_result": None,
        "step3_result": None,
        "confidence_score": 0,
        "analysis_summary": "",
        "evidence": []
    }
    
    print(f"\n{'='*60}\n🔍 트래킹 분석 시작: {image_path}\n{'='*60}")
    
    try:
        final_state = agent_app.invoke(initial_state)
        return {
            "confidence_score": final_state["confidence_score"],
            "analysis_summary": final_state["analysis_summary"],
            "step1_result": final_state["step1_result"],
            "step2_result": final_state["step2_result"],
            "step3_result": final_state["step3_result"],
            "evidence": final_state["evidence"]
        }
    except Exception as e:
        print(f"❌ 분석 중 오류 발생: {e}")
        return {"error": str(e)}

✅ Langgraph StateGraph 구성 완료


In [6]:
def test_single_step(step_name: Literal["step1", "step2", "step3"], image_path: str, prev_state: Optional[AgentState] = None):
    """
    특정 단계만 독립적으로 테스트하기 위한 함수
    """
    print(f"\n🧪 [Test] {step_name} 독립 실행 테스트 중...")
    
    if prev_state:
        state = prev_state.copy()
    else:
        state: AgentState = {
            "image_path": image_path,
            "image_part": None, # 노드 내부에서 생성됨
            "step1_result": None,
            "step2_result": None,
            "step3_result": None,
            "confidence_score": 0,
            "analysis_summary": "",
            "evidence": []
        }
    
    try:
        if step_name == "step1":
            result_state = step1_dendritic_pattern(state)
            print("결과:", json.dumps(result_state["step1_result"], indent=2, ensure_ascii=False))
        elif step_name == "step2":
            result_state = step2_luster_detection(state)
            print("결과:", json.dumps(result_state["step2_result"], indent=2, ensure_ascii=False))
        elif step_name == "step3":
            result_state = step3_surface_erosion(state)
            print("결과:", json.dumps(result_state["step3_result"], indent=2, ensure_ascii=False))
        return result_state
    except Exception as e:
        print(f"❌ 테스트 실패: {e}")
        return None

In [7]:
if __name__ == "__main__":
    # 테스트할 이미지 경로 설정 (노트북 기준 상대 경로)
    TEST_IMAGE_PATH = "../data/Cu2O_Breeding.jpg"
    
    # 이미지 파일 존재 여부 확인
    if not os.path.exists(TEST_IMAGE_PATH):
        print(f"⚠️ 경고: 테스트 이미지 '{TEST_IMAGE_PATH}'가 없습니다. 경로를 확인하세요.")
    else:
        # 전체 분석 실행
        result = analyze_tracking(TEST_IMAGE_PATH)
        
        # 결과 출력
        print("\n" + "="*60)
        print("📊 분석 결과")
        print("="*60)
        print(result.get("analysis_summary", ""))
        print(f"\n신뢰도 점수: {result.get('confidence_score', 0)}%")
        print("\n증거:")
        for ev in result.get("evidence", []):
            print(f"  - Step {ev.get('step')}: {ev.get('evidence')}")


🔍 트래킹 분석 시작: ../data/Cu2O_Breeding.jpg

🔍 [Step 1] 수지상 도전로 패턴 분석 시작...

🔍 [Step 1] 응답 파트 개수: 1

📊 [Step 1] 응답 객체 속성:
  - Candidate 속성: avg_logprobs, citation_metadata, content, finish_message, finish_reason, from_dict, function_calls, grounding_metadata, index, logprobs_result...
📋 [Step 1] Finish reason: 1

💭 [Step 1] 모델 응답 텍스트:
------------------------------------------------------------
📝 [Thinking 과정]:
## 분석 보고서

### Step 1. 객관적 관찰 (Observations)

*   이미지에는 녹색 배경 위에 두 개의 구리 전선이 놓여 있습니다. 하나는 여러 가닥이 꼬인 굵은 전선이고, 다른 하나는 단일 가닥으로 보이는 얇은 전선입니다.
*   **굵은 전선:**
    *   오른쪽 끝부분의 황토색 외부 피복이 파괴되어 있으며, 내부의 구리선이 노출되어 있습니다. 노출된 구리선 주변의 내부 절연체는 검게 탄화되고 부서져 있습니다.
    *   왼쪽 끝부분에는 녹색의 불명확한 부품이 있으며, 이 주변은 검은색 물질로 덮여 있고 일부 용융된 것처럼 보입니다. 주변에 미세한 흰색 가루가 흩어져 있습니다.
*   **얇은 전선:**
    *   전선 중앙 부분에 약 2~3cm 길이의 구간이 검게 변색 또는 탄화되어 있습니다.
*   **패턴 분석:**
    *   전선 피복의 손상 부위에서 나뭇가지(Dendrite)처럼 뻗어 나가는 미세하고 복잡한 선형 패턴은 관찰되지 않습니다.
    *   손상은 특정 부위에 집중된 광범위한 탄화 및 용융, 또는 전선 경로를 따른 선형적인 탄화의 형태를 보입니다.
    *   두 전선(굵은 